# OSRT Ostinato — midtrain v4: steps 13,500 → 31,500, then a 2,000-step cooldown

Target runtime: **RTX PRO 6000 Blackwell, 96 GB** (Colab Pro+, ~8 units/h).

The plan this notebook runs (docs/PROGRESS.md §9f, decided 2026-11-06):

| stage | cell | from → to | ~hours | repo |
|---|---|---|---|---|
| stable phase, LR 5e-5 | §5 | 13,500 → 31,500 | 125 | `HallD/OSRT-Ostinato-midtrain-v4` |
| cooldown, 5e-5 → 5e-6 | §7 | 31,500 → 33,500 | 14 | `HallD/OSRT-Ostinato-midtrain-v4-cooldown-31500` |

Lineage after the stable phase ≈ 15.4B tokens (~2.9× Chinchilla on the 263M
active params). The 14,500 cooldown branch and its SFT merge stay in
`-v4-cooldown` as a leaf; nothing trains on from a cooled checkpoint.

Run the cells in order. Sessions are capped and the VM's disk is wiped on
release: §5 pulls the newest checkpoint from the mirror before training and
pushes each new one, so **when the session dies, re-run §5** — it resumes.
Do **not** background the training cell (`nohup`, `&`): Colab tears the
runtime down when the foreground cell returns.


## 1 · Gate: what card is this, and can the expert path go low-precision?

This is also the first half of roadmap gate **G7**. Routed experts are ~84%
of v7's parameters and run through the private `torch._grouped_mm`; if that
refuses FP8, the NVFP4 case in §13.3 covers under a third of the model.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv


## 2 · Install

The resolver's output is left visible on purpose — a silent downgrade here is
how a run dies an hour in. The gate below refuses **torch < 2.10** or
**transformers < 5.3** (the pins `app.py`'s Modal image carries): torch 2.8 is
known to fail Inductor's C++ codegen on this model's compiled forward.


In [ ]:
!git clone -q https://github.com/CodeHalwell/OSRT-Ostinato.git
%cd OSRT-Ostinato
# Same pins as the Modal image in app.py. Colab's preinstalled torch and
# transformers satisfy pyproject's floors, so a bare `pip install -e .` would
# leave them in place and the version check in the next cell would abort.
!pip install -q torch==2.10.0+cu128 --index-url https://download.pytorch.org/whl/cu128
# Colab's preinstalled torchvision/torchaudio were built against ITS torch; left in
# place they break `import transformers` (torchvision::nms does not exist).
!pip uninstall -q -y torchvision torchaudio
!pip install -q transformers==5.3.0 datasets==4.6.1 tokenizers==0.22.2
!pip install -q safetensors==0.7.0 "huggingface_hub>=0.35" wandb==0.25.1
!pip install -q -e .
!PYTHONPATH=src python scripts/probe_gpu.py


In [ ]:
# Same floor as the Modal image in app.py (torch 2.10.0+cu128, transformers
# 5.3.0). torch 2.8 dies in Inductor codegen ("'zuf3' was not declared in this
# scope") at the first compiled forward, so refuse before spending anything.
import torch
import transformers
from packaging.version import Version

assert Version(torch.__version__.split('+')[0]) >= Version('2.10'), torch.__version__
assert Version(transformers.__version__) >= Version('5.3'), transformers.__version__
print('torch', torch.__version__, '| transformers', transformers.__version__)


## 3 · Secrets

Add `HF_TOKEN` (write access to your checkpoint repo) and `WANDB_API_KEY`
in the Colab **Secrets** panel (🔑, left sidebar). They never touch this
notebook or git.


In [ ]:
import os

from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
assert os.environ['HF_TOKEN'] and os.environ['WANDB_API_KEY'], 'set both in Secrets'
print('secrets set')


## 4 · Check the tokenizer and confirm the shape

The v7 tokenizer is **committed** in `tokenizer/` (SmolLM2 + 32 OSRT specials,
roadmap §16). Do not rebuild it per session: a rebuild that drifted by one
token would silently invalidate every checkpoint. The contract check pins the
vocab size and every structural token id, and is the same check the trainer
runs before building the model.

`compute_budget.py` is the only trusted source for parameter counts — the
repo deliberately states none in any name. Expect **968,468,355 physical /
263,035,779 active**; anything else means the config drifted.


In [ ]:
%%bash
# Tokenizer contract + the canonical parameter counts, before any GPU spend.
PYTHONPATH=src python - <<'EOF'
from transformers import AutoTokenizer
from osrt.tokenizer_contract import validate_tokenizer_contract
validate_tokenizer_contract(AutoTokenizer.from_pretrained('tokenizer'))
print('tokenizer contract OK')
EOF
PYTHONPATH=src python scripts/compute_budget.py


## 5 · The stable phase: 13,500 → 31,500

Continues **midtrain v4** (`MidtrainConfig`: WSD, peak 5e-5, the widened
24-source mix) from the mirror's newest `osrt_step_N.pt` (weights + optimizer +
data cursors). Every new checkpoint is pushed back every 500 steps.

`--stop-step 31500` ends the cell by itself once `osrt_step_31500.pt` is
saved and flushed to the mirror. The LR schedule is untouched (total steps
stay 123,000, so the LR is 5e-5 throughout), which is what lets the cooldown
branch off a true plateau checkpoint. If a restart overshoots to 32,000, just
set `FROM_STEP = 32000` in §7.

> **Card size.** 270,336 tokens/step are 6 × 4096 × 11 on a B200;
> `--micro-batch-scale 0.5` makes that 3 × 4096 × 22 here — same tokens per
> step, same run. Expect ~11K tok/s steady (25 s/step), 90 GB VRAM.

> **W&B.** The run id travels in `wandb_run_id.txt` on the mirror, so this
> continues W&B run `osrt-v7-midtrain-v4`. The client buffers for up to two
> hours and the dashboard shows "crashed" meanwhile — trust the cell output.


In [ ]:
# sm_120 (this card) caps shared memory at 99 KB/block; Inductor's mix-order
# fused-RMSNorm-backward kernel needs ~120 KB. Read by torch at import, so it
# must be in the environment of the training process, not set inside it.
%env TORCHINDUCTOR_MIX_ORDER_REDUCTION=0
%env TORCHINDUCTOR_PERSISTENT_REDUCTIONS=0
# The grouped-GEMM experts fragment the caching allocator (68 -> 90 GB in 50
# steps); expandable segments reclaim that on the 96 GB card.
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
HF_CKPT_REPO = 'HallD/OSRT-Ostinato-midtrain-v4'   # private mirror of the v4 run
STOP_STEP = 31500

!PYTHONPATH=src python -m osrt.train_main \
    --config midtrain \
    --tokenizer-path ./tokenizer \
    --ckpt-dir ./checkpoints/midtrain_v4 \
    --hf-repo {HF_CKPT_REPO} \
    --micro-batch-scale 0.5 \
    --stop-step {STOP_STEP}


## 6 · What to watch

The health check runs on **every log step** (no longer only at step 5,000)
and ends the run on router collapse, loop collapse, residual explosion or dead
data — that is the design. The last line of the cell names the outcome:
`early_stop` (the log names the criterion; `osrt_failed_step_N.pt` holds the
state and is not resumed), `rescued` (time boundary — re-run the cell),
`data_dead` (every source failed for good — fix the data, re-run), `complete`.
Otherwise:

| signal | healthy | worry |
|---|---|---|
| `loss` | falling, no unrecovered spikes | flat, or spiking |
| `train/grad_norm` | settles after warmup, spikes recover | climbing, or a staircase |
| `moe/dead_experts_total` | 0 | > 0 — at E=28 each is 3.6% of a block |
| `moe/marginal_entropy_mean` | near ln(28) ≈ 3.33 | falling — router collapse |
| `moe/prebias_expert_min_mean` | clearly > 0 | → 0 while loss still falls: the balance bias is carrying a dead router |
| `moe/b*/bias_loop_spread` | flat | rising — routing diverging across loops |
| `loop/update_norm_l*` | non-trivial at every loop | last loops → 0 = loop collapse |
| `loop/hidden_norm_ratio` | ≈ 1, flat | rising — residual explosion (§17.3) |
| `muon/ortho_err` | small, flat | rising — Newton–Schulz not converging |
| `train/dead_sources` | 0 | > 0 — a data source failed permanently |

The bets this run embodies, and what would falsify each, are in roadmap §19.


## 7 · Cooldown: 31,500 → 33,500

Branches off `osrt_step_31500.pt` with its optimizer state; the run ends
`COOLDOWN_STEPS` later with the LR decaying linearly 5e-5 → 5e-6 over exactly
those steps (`train_main --cooldown-steps`). Evals and checkpoints every 500
steps. The first 1,000-step cooldown (13,500 → 14,500) was worth 0.19 nats
of held-out loss; 2,000 steps is the longer version of the same thing.

It pushes to a **fresh** mirror: the push daemon keeps only the three newest
step files, so reusing `-v4-cooldown` would prune the 14,000/14,500 branch.
The main mirror's "latest" stays the plateau checkpoint. Re-run this cell
if the session dies; it resumes from the cooldown mirror.

**Afterwards:** compare held-out at 33,500 against 14,500's 2.9375. That
checkpoint is the base for the full-weight SFT (to be added here), and the
HRA probe (`scripts/run_sft.py --micro-batch-scale 0.25`) runs on the
full-SFT merge, not on the base.


In [ ]:
# sm_120 (this card) caps shared memory at 99 KB/block; Inductor's mix-order
# fused-RMSNorm-backward kernel needs ~120 KB. Read by torch at import, so it
# must be in the environment of the training process, not set inside it.
%env TORCHINDUCTOR_MIX_ORDER_REDUCTION=0
%env TORCHINDUCTOR_PERSISTENT_REDUCTIONS=0
# The grouped-GEMM experts fragment the caching allocator (68 -> 90 GB in 50
# steps); expandable segments reclaim that on the 96 GB card.
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
FROM_STEP = 31500                                  # <-- plateau checkpoint to branch from
COOLDOWN_STEPS = 2000
COOLDOWN_REPO = 'HallD/OSRT-Ostinato-midtrain-v4-cooldown-31500'   # private; created on first push

from huggingface_hub import hf_hub_download
hf_hub_download('HallD/OSRT-Ostinato-midtrain-v4', f'osrt_step_{FROM_STEP}.pt',
                local_dir='./checkpoints/cooldown')

!PYTHONPATH=src python -m osrt.train_main \
    --config midtrain \
    --tokenizer-path ./tokenizer \
    --ckpt-dir ./checkpoints/cooldown \
    --hf-repo {COOLDOWN_REPO} \
    --cooldown-steps {COOLDOWN_STEPS} \
    --micro-batch-scale 0.5 \
    --wandb-run-name osrt-v7-midtrain-v4-cooldown-31500
